# Reproducibility smoke test on the free sample

This notebook runs `../validation/benchmark_script.py`, **unmodified and identical to the script used for the published benchmark**, against the free sample shipped in `../sample/`.

**Purpose:** show that the code runs end to end against the real delivered schema.

**Not the purpose:** reproduce the published metric. The sample holds 40 of the 10,000 subjects (120 subject-branch rows). With 5-fold cross-validation each test fold has 24 rows, so fold-level accuracy and AUC move by several points between folds.

The number to cite is the one measured on the full dataset: Task A (binary high VGE, logistic regression, 5-fold CV) AUC 0.7516, accuracy 73.87%; Task B (5-class VGE grade, gradient boosting) accuracy 41.8%, weighted kappa 0.344 (n = 10,000 subjects x 3 branches).


In [1]:
import sys, subprocess, shutil, tempfile, pathlib
import pandas as pd
HERE = pathlib.Path.cwd()
SAMPLE = (HERE / ".." / "sample").resolve()
SCRIPT = (HERE / ".." / "validation" / "benchmark_script.py").resolve()
work = pathlib.Path(tempfile.mkdtemp(prefix="smoke_"))   # all outputs go here, nothing is written into the repository
print("sample dir :", SAMPLE.name)
print("script     :", SCRIPT.name, "(unmodified copy of the published benchmark script)")
def clean(text):
    return text.replace(str(work), "<tmp>").replace(str(SAMPLE), "<sample>")


sample dir : sample
script     : benchmark_script.py (unmodified copy of the published benchmark script)


In [2]:
import importlib.util
spec = importlib.util.spec_from_file_location("benchmark_script", SCRIPT)
benchmark_script = importlib.util.module_from_spec(spec)
spec.loader.exec_module(benchmark_script)
benchmark_script.DATASET = str(SAMPLE)
benchmark_script.OUT = str(work)
summary = benchmark_script.main()

                  task           fold                       model  n_test  accuracy       auc  weighted_kappa
0    A_high_vge_binary              0          LogisticRegression      24  0.666667  0.647059             NaN
1    A_high_vge_binary              1          LogisticRegression      24  0.708333  0.705882             NaN
2    A_high_vge_binary              2          LogisticRegression      24  0.666667  0.644531             NaN
3    A_high_vge_binary              3          LogisticRegression      24  0.625000  0.648438             NaN
4    A_high_vge_binary              4          LogisticRegression      24  0.666667  0.554688             NaN
5    A_high_vge_binary  bayes_ceiling       true_cell_probability     120  0.700000  0.673299             NaN
6   B_vge_grade_5class              0  GradientBoostingClassifier      24  0.375000       NaN        0.244019
7   B_vge_grade_5class              1  GradientBoostingClassifier      24  0.416667       NaN        0.267857
8   B_vge_

## Reading the output

Task A should land near the 'bayes_ceiling' row printed with it: in this dataset the binary label is fully determined by branch group and PFO status, so the ceiling is a property of the construction. Task B is the honest weak result: the fine grade split is a random draw, so five-class accuracy stays low. In the executed run Task A reaches a mean AUC of 0.646 (folds 0.555 to 0.706) against a ceiling of 0.673 on these 120 rows, and Task B a mean accuracy of 0.358 with weighted kappa between -0.165 and 0.268 across folds.

In [3]:
results = pd.read_parquet(work / "benchmark_results.parquet")
results

,task,fold,model,n_test,accuracy,auc,weighted_kappa
0,A_high_vge_binary,0,LogisticRegression,24,0.666667,0.647059,NaN
1,A_high_vge_binary,1,LogisticRegression,24,0.708333,0.705882,NaN
2,A_high_vge_binary,2,LogisticRegression,24,0.666667,0.644531,NaN
3,A_high_vge_binary,3,LogisticRegression,24,0.625000,0.648438,NaN
4,A_high_vge_binary,4,LogisticRegression,24,0.666667,0.554688,NaN
5,A_high_vge_binary,bayes_ceiling,true_cell_probability,120,0.700000,0.673299,NaN
6,B_vge_grade_5class,0,GradientBoostingClassifier,24,0.375000,NaN,0.244019
7,B_vge_grade_5class,1,GradientBoostingClassifier,24,0.416667,NaN,0.267857
8,B_vge_grade_5class,2,GradientBoostingClassifier,24,0.250000,NaN,-0.165468
9,B_vge_grade_5class,3,GradientBoostingClassifier,24,0.375000,NaN,0.000000
